# NB_hospital_daily_summary

Builds `adb_caresync.gold.hospital_daily_summary` — one row per hospital per calendar day with test volumes, abnormal/critical rates, and day-over-day trend. Full rebuild via CREATE OR REPLACE TABLE.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.gold.hospital_daily_summary AS

WITH daily_agg AS (
    SELECT
        lr.hospital_id,
        lr.test_date                                                        AS summary_date,
        COUNT(*)                                                            AS total_tests,
        COUNT(DISTINCT lr.patient_id)                                       AS unique_patients_tested,
        COUNT(DISTINCT lr.doctor_id)                                        AS active_doctor_count,
        COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)          AS abnormal_test_count,
        COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)          AS critical_test_count
    FROM adb_caresync.silver.lab_results lr
    GROUP BY lr.hospital_id, lr.test_date
),

with_window AS (
    SELECT
        da.*,
        CAST(da.abnormal_test_count * 100.0 / da.total_tests AS DECIMAL(5,2))   AS abnormal_rate_pct,
        CAST(da.critical_test_count * 100.0 / da.total_tests AS DECIMAL(5,2))   AS critical_rate_pct,
        LAG(da.total_tests) OVER (PARTITION BY da.hospital_id ORDER BY da.summary_date)
            AS prev_day_total_tests
    FROM daily_agg da
)

SELECT
    w.hospital_id,
    h.hospital_name,
    w.summary_date,
    c.day_name,
    c.is_weekend,
    w.total_tests,
    w.unique_patients_tested,
    w.active_doctor_count,
    w.abnormal_test_count,
    w.critical_test_count,
    w.abnormal_rate_pct,
    w.critical_rate_pct,
    w.prev_day_total_tests,
    CAST(
        CASE
            WHEN w.prev_day_total_tests IS NULL OR w.prev_day_total_tests = 0 THEN NULL
            ELSE (w.total_tests - w.prev_day_total_tests) * 100.0 / w.prev_day_total_tests
        END
    AS DECIMAL(6,2)) AS test_volume_change_pct
FROM with_window w
LEFT JOIN adb_caresync.silver.hospitals h
    ON w.hospital_id = h.hospital_id
LEFT JOIN adb_caresync.silver.calendar c
    ON c.date_key = CAST(DATE_FORMAT(w.summary_date, 'yyyyMMdd') AS INT)

In [0]:
record_count = spark.table("adb_caresync.gold.hospital_daily_summary").count()
dbutils.notebook.exit(str(record_count))